In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import datetime
import os
import re
import time

import pandas as pd
import pdfplumber
import requests


from time import sleep

import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'TN CBTUN'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running TN CBTUN Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# Chrome driver not needed — PDF downloaded via requests + parsed with pdfplumber.
# Kept as placeholder in case Selenium is needed for future list types.
# chromeOptions = webdriver.ChromeOptions()
# prefs = {
#     'plugins.always_open_pdf_externally': True,
#     'download.prompt_for_download': False,
#     'download.default_directory': tempfolder,
#     'profile.default_content_setting_values.automatic_downloads': 1,
# }
# chromeOptions.add_experimental_option('prefs', prefs)
# driver = webdriver.Chrome(options=chromeOptions)
# driver.maximize_window()

In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------

processdate = now.strftime('%Y-%m-%d')

PDF_URL = 'https://www.bct.gov.tn/bct/siteprod/documents/20240314_Structure_Systeme_Financier_fr.PDF'

regdict = {
    regulatorName + ' 1': PDF_URL,
}

Typology = {
    regulatorName + ' 1': 'Banking Supervision',
}

SECTION_TYPOLOGY = {
    'BANQUES RESIDENTES':                'List of Resident Banks',
    'BANQUES NON RESIDENTES':            'List of Non-Resident Banks',
    'ETABLISSEMENTS DE LEASING':         'List of Leasing Institutions',
    'SOCIETES DE FACTORING':             'List of Factoring Companies',
    'BANQUES D\'AFFAIRES':               'List of Investment Banks',
    'BUREAUX DE REPRESENTATION':         'List of Representative Offices',
    'ETABLISSEMENTS DE PAIEMENT':        'List of Payment Institutions',
}

sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [],
    'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
    'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [],
    'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [],
    'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [],
    'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [],
    'CancellationDate': [], 'RegCtry': [], 'RegCode': [], 'ListCode': [],
    'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [],
    'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
    'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [],
    'Cntry - Mother company': [], 'Phone - Mother company': [], 'Check': [],
}

In [5]:
#------------------------------------------------ Begin_Function ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] = sqldict[key] + [''] * (maxlen - len(sqldict[key]))
    return sqldict


def download_pdf(url, dest_folder):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/143.0.0.0 Safari/537.36"
    }
    resp = requests.get(url, headers=headers, timeout=60, verify=False)
    resp.raise_for_status()
    fname = url.split('/')[-1]
    fpath = os.path.join(dest_folder, fname)
    with open(fpath, 'wb') as f:
        f.write(resp.content)
    print(f"[INFO] Downloaded {fname} ({len(resp.content)//1024} KB)")
    return fpath


def detect_section(text):
    """Return the SECTION_TYPOLOGY key if this page is a section header, else None."""
    upper = text.upper().strip()
    for key in SECTION_TYPOLOGY:
        if key in upper:
            return key
    return None


def _extract_field(text, labels):
    """Pull the value after 'label :' trying each label variant in order."""
    if isinstance(labels, str):
        labels = [labels]
    for label in labels:
        pattern = re.compile(
            rf'-\s*{re.escape(label)}\s*:\s*(.*)',
            re.IGNORECASE
        )
        m = pattern.search(text)
        if m and m.group(1).strip():
            return m.group(1).strip()
    return ''


def parse_entity_page(text):
    """Parse a single entity page and return a dict of fields, or None if not an entity page."""
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) < 3:
        return None

    has_code = bool(re.search(r'Code\s+banque', text, re.IGNORECASE))
    has_bcant = bool(re.search(r'Banque\s+de\s+cantonnement', text, re.IGNORECASE))
    has_addr = bool(re.search(r'Adresse\s*(du\s+si\xe8ge\s+social\s*)?:', text, re.IGNORECASE))
    if not (has_code or has_bcant or has_addr):
        return None

    name = lines[0]
    if name.upper().startswith(('BANQUES ', 'ETABLISSEMENT', 'BUREAU', 'STRUCTURE')):
        if len(lines) > 1 and (has_code or has_bcant):
            pass
        else:
            return None

    code_match = re.search(r'Code\s+banque\s*:\s*(\S+)', text, re.IGNORECASE)
    code_banque = code_match.group(1) if code_match else ''

    bcant = _extract_field(text, 'Banque de cantonnement')

    address_raw = _extract_field(text, ['Adresse du si\xe8ge social', 'Adresse'])
    city = ''
    zipcode = ''
    if address_raw:
        zip_m = re.search(r'\b(\d{4})\b', address_raw)
        if zip_m:
            zipcode = zip_m.group(1)
        parts = re.split(r',\s*', address_raw)
        if len(parts) >= 2:
            last_part = parts[-1].strip()
            city_m = re.sub(r'\d{4}', '', last_part).strip()
            if city_m:
                city = city_m

    website = _extract_field(text, ['Adresse du site web', 'Adresse de site web', 'Site web', 'Adresse de site'])

    phone_raw = _extract_field(text, ['T\xe9l\xe9phone', 'Tel'])

    fax_raw = _extract_field(text, ['T\xe9l\xe9fax', 'Fax'])

    email = _extract_field(text, ['E-mail', 'Email', 'Adresse e-mail'])

    return {
        'name': name,
        'code_banque': code_banque,
        'banque_cantonnement': bcant,
        'address': address_raw,
        'city': city,
        'zip': zipcode,
        'website': website,
        'phone': phone_raw,
        'fax': fax_raw,
        'email': email,
    }

In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f'[INFO] : Working {k+1}/{len(regdict)} _({reg})_')

    pdf_path = download_pdf(regdict[reg], tempfolder)
    current_section = 'Banking Supervision'
    entity_count = 0

    with pdfplumber.open(pdf_path) as pdf:
        print(f"[INFO] PDF has {len(pdf.pages)} pages")
        for page_num, page in enumerate(pdf.pages):
            text = page.extract_text() or ''
            if not text.strip():
                continue

            section = detect_section(text)
            if section:
                current_section = SECTION_TYPOLOGY[section]
                print(f"[INFO] Section detected (p.{page_num+1}): {current_section}")

            entity = parse_entity_page(text)
            if entity is None:
                continue

            entity_count += 1
            sqldict['Name'].append(entity['name'])
            sqldict['InternalID_1'].append(entity['code_banque'])
            sqldict['InternalID_1_type'].append('Code banque' if entity['code_banque'] else '')
            sqldict['License_Type'].append(entity.get('banque_cantonnement', ''))
            sqldict['Address_1'].append(entity['address'])
            sqldict['City'].append(entity['city'])
            sqldict['Zip'].append(entity['zip'])
            sqldict['Phone'].append(entity['phone'])
            sqldict['Fax'].append(entity['fax'])
            sqldict['Website'].append(entity['website'])
            sqldict['Email'].append(entity['email'])
            sqldict['Cntry'].append('TN')
            sqldict['RegCtry'].append('TN')
            sqldict['RegCode'].append('CBTUN')
            sqldict['ListCode'].append(reg.split()[-1])
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldict['Typology'].append(current_section)
            sqldict['ListProcessDate'].append(processdate)

            sqldict = bourange_same_length_array(sqldict)

    print(f"[INFO] Extracted {entity_count} entities from PDF")

    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))

[INFO] : Working 1/1 _(TN CBTUN 1)_


[INFO] Downloaded 20240314_Structure_Systeme_Financier_fr.PDF (819 KB)
[INFO] PDF has 61 pages
[INFO] Section detected (p.2): List of Resident Banks
[INFO] Section detected (p.26): List of Leasing Institutions
[INFO] Section detected (p.35): List of Factoring Companies
[INFO] Section detected (p.38): List of Investment Banks
[INFO] Section detected (p.49): List of Representative Offices
[INFO] Section detected (p.56): List of Payment Institutions
[INFO] Extracted 53 entities from PDF


In [7]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, sheet_name='SQL Ready', index=False)
sleep(2)
print(f"[INFO] Excel file '{filename}' saved — {len(df)} rows")

[INFO] Excel file 'TN CBTUN SQL Ready 2026-04-21 14.51.14.xlsx' saved — 53 rows
